# BrailleLens — YOLO26 Dot Detection (Colab GPU)

Fine-tunes **YOLO26** (`yolo26n.pt`) on DSBI Braille raised-dot boxes.

**Disconnect-safe:** every epoch checkpoint is written to Google Drive under  
`MyDrive/BrailleLens_YOLO26/runs/...`. Re-run the Train cell to resume from `last.pt`.

### Before you start
1. Upload `braille_dots.zip` → `MyDrive/BrailleLens_YOLO26/braille_dots.zip`  
2. **Runtime → Change runtime type → GPU**  
3. Run cells in order  

See `yolo_dot_detect/COLAB_SETUP.md` for full path list.

## 0) Optional — reduce idle disconnects
Run this once. It does **not** replace Drive checkpoints; it only pings the browser.

In [ ]:
from IPython.display import Javascript, display
display(Javascript('''
function ClickConnect(){
  console.log("Keepalive ping");
  const btn = document.querySelector("colab-connect-button")
          || document.querySelector("#top-toolbar colab-connect-button");
  if (btn) btn.click();
}
setInterval(ClickConnect, 60000);
'''))
print("Keepalive JS registered (60s). Weights still save to Drive every epoch.")

## 1) Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path

# ===== Drive layout (do not rename unless you change upload paths) =====
DRIVE_ROOT = Path("/content/drive/MyDrive/BrailleLens_YOLO26")
ZIP_PATH   = DRIVE_ROOT / "braille_dots.zip"
DATA_ROOT  = DRIVE_ROOT / "datasets" / "braille_dots"
DATA_YAML  = DATA_ROOT / "data.yaml"
RUNS_DIR   = DRIVE_ROOT / "runs" / "detect"
# New name on purpose: the earlier full-page run stored imgsz=1280 args in its
# last.pt, and resuming that would override the tiled settings.
RUN_NAME   = "braille_dot_yolo26_tiled"
WEIGHTS_DIR = RUNS_DIR / RUN_NAME / "weights"
LAST_PT    = WEIGHTS_DIR / "last.pt"
BEST_PT    = WEIGHTS_DIR / "best.pt"

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print("DRIVE_ROOT :", DRIVE_ROOT)
print("ZIP exists :", ZIP_PATH.exists(), "->", ZIP_PATH)
print("DATA_ROOT  :", DATA_ROOT)

## 2) Install YOLO26 stack
Needs a recent `ultralytics` that ships YOLO26 weights (`yolo26n.pt`).

**If the runtime restarts here, that is expected** — Colab's preinstalled Pillow
conflicts with the version Ultralytics needs. After the restart, re-run cell 1
(Mount Drive) and then this cell again.

In [ ]:
import os

# Pillow must be reinstalled cleanly: upgrading it in-place leaves mixed
# old/new files and breaks `from PIL import ImageText` until the process restarts.
!pip -q uninstall -y pillow
!pip -q install --no-cache-dir --force-reinstall "pillow>=11.3.0"
!pip -q install -U ultralytics pyyaml opencv-python-headless

try:
    from PIL import ImageText  # noqa: F401
    from ultralytics import YOLO  # noqa: F401

    import torch
    import ultralytics

    print("ultralytics", ultralytics.__version__)
    print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
    else:
        print("WARNING: no GPU — Runtime → Change runtime type → GPU")
    print("\nInstall OK — continue to the next cell.")
except ImportError as e:
    print("Import failed:", e)
    print("\nRestarting runtime to load the new Pillow...")
    print("After it restarts, re-run cell 1 (Mount Drive) and this cell again.")
    os.kill(os.getpid(), 9)

## 3) Unpack to local disk (fast)

Expects zip at `MyDrive/BrailleLens_YOLO26/braille_dots.zip`.

Unpacks to **`/content/braille_dots`** rather than Drive. Training off mounted
Drive triggers `Slow image access` warnings and drags every epoch; local disk is
many times faster. Only checkpoints go to Drive.

In [ ]:
import zipfile
import yaml

# Local (fast) working copy — regenerate freely, it is disposable
LOCAL_PAGES = Path("/content/braille_dots")
LOCAL_PAGES_YAML = LOCAL_PAGES / "data.yaml"

if LOCAL_PAGES_YAML.exists():
    print("Pages already unpacked locally:", LOCAL_PAGES)
else:
    if not ZIP_PATH.exists():
        raise FileNotFoundError(
            f"Missing {ZIP_PATH}\n"
            "Upload braille_dots.zip to MyDrive/BrailleLens_YOLO26/ first.\n"
            "On PC: py -3.11 -m yolo_dot_detect.pack_for_colab"
        )
    print("Unpacking", ZIP_PATH, "-> /content ...")
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall("/content")
    print("Unpacked to", LOCAL_PAGES)

cfg = {
    "path": str(LOCAL_PAGES.resolve()),
    "train": "images/train",
    "val": "images/test",
    "names": {0: "braille_dot"},
    "nc": 1,
}
with open(LOCAL_PAGES_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

n_train = len(list((LOCAL_PAGES / "images" / "train").glob("*.jpg")))
n_val = len(list((LOCAL_PAGES / "images" / "test").glob("*.jpg")))
print(f"full pages -> train: {n_train} | val: {n_val}")

## 3b) Tile the pages — **required fix for the CUDA OOM**

A full DSBI page carries 1,000–5,000 dots. YOLO's label assigner allocates memory
proportional to *boxes × anchors*, which is what produced:

```
WARNING ⚠️ CUDA OutOfMemoryError in TaskAlignedAssigner, using CPU
```

Falling back to CPU for assignment makes training crawl. Slicing each page into
640px tiles drops it to ~50–150 dots per image, and dots keep their native pixel
size instead of shrinking during resize — which also fixes the very low mAP from
the earlier full-page run.

Takes a couple of minutes; skipped if already tiled.

In [ ]:
from PIL import Image

Image.MAX_IMAGE_PIXELS = None

TILE = 640          # tile size in pixels (also use as imgsz when training)
OVERLAP = 96        # overlap so dots on seams are not lost
MIN_BOXES = 5       # discard near-empty tiles
VAL_PAGES = 40      # cap val pages to keep per-epoch validation quick

TILED = Path("/content/braille_dots_tiled")
TILED_YAML = TILED / "data.yaml"


def _origins(total, tile, stride):
    if total <= tile:
        return [0]
    xs = list(range(0, total - tile + 1, stride))
    if xs[-1] != total - tile:
        xs.append(total - tile)
    return xs


def tile_split(src, dst, split, max_pages=None):
    src_img, src_lbl = src / "images" / split, src / "labels" / split
    dst_img, dst_lbl = dst / "images" / split, dst / "labels" / split
    dst_img.mkdir(parents=True, exist_ok=True)
    dst_lbl.mkdir(parents=True, exist_ok=True)

    pages = sorted(src_img.glob("*.jpg"))
    if max_pages:
        pages = pages[:max_pages]
    stride = max(TILE - OVERLAP, 1)
    n_tiles = n_boxes = 0

    for p in pages:
        lbl = src_lbl / (p.stem + ".txt")
        if not lbl.exists():
            continue
        rows = []
        for line in lbl.read_text().splitlines():
            f = line.split()
            if len(f) >= 5:
                rows.append((int(float(f[0])), *(float(v) for v in f[1:5])))
        if not rows:
            continue

        with Image.open(p) as im:
            im = im.convert("RGB")
            pw, ph = im.size
            absb = [(c, xc * pw, yc * ph, w * pw, h * ph) for c, xc, yc, w, h in rows]

            for oy in _origins(ph, TILE, stride):
                for ox in _origins(pw, TILE, stride):
                    tw, th = min(TILE, pw), min(TILE, ph)
                    keep = []
                    for c, acx, acy, aw, ah in absb:
                        if not (ox <= acx < ox + tw and oy <= acy < oy + th):
                            continue
                        x0 = max(acx - aw / 2 - ox, 0.0)
                        y0 = max(acy - ah / 2 - oy, 0.0)
                        x1 = min(acx + aw / 2 - ox, float(tw))
                        y1 = min(acy + ah / 2 - oy, float(th))
                        if x1 - x0 < 1 or y1 - y0 < 1:
                            continue
                        keep.append(
                            f"{c} {(x0 + x1) / 2 / tw:.6f} {(y0 + y1) / 2 / th:.6f} "
                            f"{(x1 - x0) / tw:.6f} {(y1 - y0) / th:.6f}"
                        )
                    if len(keep) < MIN_BOXES:
                        continue
                    stem = f"{p.stem}__x{ox}_y{oy}"
                    im.crop((ox, oy, ox + tw, oy + th)).save(
                        dst_img / f"{stem}.jpg", quality=92
                    )
                    (dst_lbl / f"{stem}.txt").write_text("\n".join(keep) + "\n")
                    n_tiles += 1
                    n_boxes += len(keep)
    return len(pages), n_tiles, n_boxes


if TILED_YAML.exists():
    print("Already tiled:", TILED)
else:
    for split, limit in (("train", None), ("test", VAL_PAGES)):
        pages, tiles, boxes = tile_split(LOCAL_PAGES, TILED, split, limit)
        per = boxes / tiles if tiles else 0
        print(f"{split:5s}: {pages:4d} pages -> {tiles:5d} tiles, {boxes:7d} boxes ({per:.0f}/tile)")

with open(TILED_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump(
        {
            "path": str(TILED.resolve()),
            "train": "images/train",
            "val": "images/test",
            "names": {0: "braille_dot"},
            "nc": 1,
        },
        f,
        sort_keys=False,
    )

print("\ntiles train:", len(list((TILED / 'images' / 'train').glob('*.jpg'))))
print("tiles val  :", len(list((TILED / 'images' / 'test').glob('*.jpg'))))
print("data.yaml  ->", TILED_YAML)

## 4) Train YOLO26 (transfer learning + augmentation)

- Base weights: **`yolo26n.pt`** (COCO pretrained)
- Trains on the **tiled** dataset at `imgsz=640` (no assigner OOM)
- Reads images from local disk, writes checkpoints to Drive every epoch
- Auto-**resume** from `last.pt` on Drive after a disconnect

Still OOM on a T4? Drop `BATCH` to 4. Do **not** raise `IMGSZ` above `TILE` —
that upscales tiles and wastes memory for no accuracy gain.

In [ ]:
from ultralytics import YOLO

# -------- hyperparams (edit here) --------
MODEL  = "yolo26n.pt"   # yolo26n / yolo26s / yolo26m ...
EPOCHS = 80
IMGSZ  = TILE           # 640 — matches tile size, keeps dots at native scale
BATCH  = 8              # T4: 8 works on tiles; drop to 4 if OOM
PATIENCE = 25
# ----------------------------------------

resume = LAST_PT.exists()
print("Resume from last.pt?", resume, "|", LAST_PT)

if resume:
    model = YOLO(str(LAST_PT))
else:
    model = YOLO(MODEL)  # downloads COCO-pretrained YOLO26 on first use

results = model.train(
    data=str(TILED_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=0 if torch.cuda.is_available() else "cpu",
    project=str(RUNS_DIR),
    name=RUN_NAME,
    exist_ok=True,
    resume=resume,
    save=True,
    save_period=1,          # every epoch -> Drive (disconnect-safe)
    patience=PATIENCE,
    workers=4,
    seed=42,
    max_det=1000,           # per tile, not per page
    plots=True,
    # --- augmentation (no flips: Braille orientation) ---
    hsv_h=0.015,
    hsv_s=0.50,
    hsv_v=0.40,
    degrees=10.0,
    translate=0.10,
    scale=0.40,
    shear=2.0,
    perspective=0.0008,
    flipud=0.0,
    fliplr=0.0,
    mosaic=0.5,             # lower than full-page: mosaic multiplies boxes 4x
    mixup=0.1,
    erasing=0.25,
    close_mosaic=15,
)

print("\nDone / paused.")
print("best.pt ->", BEST_PT, "exists:", BEST_PT.exists())
print("last.pt ->", LAST_PT, "exists:", LAST_PT.exists())

## 5) Evaluate best checkpoint on Drive

In [ ]:
from ultralytics import YOLO

if not BEST_PT.exists():
    raise FileNotFoundError(f"No best.pt yet at {BEST_PT} — finish / resume training first")

model = YOLO(str(BEST_PT))
metrics = model.val(
    data=str(TILED_YAML),
    imgsz=IMGSZ,
    device=0 if torch.cuda.is_available() else "cpu",
    conf=0.25,
    max_det=1000,
    split="val",
    plots=True,
)
box = metrics.box
print(f"Precision : {box.mp:.4f}")
print(f"Recall    : {box.mr:.4f}")
print(f"mAP50     : {box.map50:.4f}")
print(f"mAP50-95  : {box.map:.4f}")

## 6) Quick infer on a full page (optional)

Upload a test photo to `MyDrive/BrailleLens_YOLO26/test_images/`.

The model was trained on 640px tiles, so a full page is **sliced into tiles**,
predicted per tile, then boxes are mapped back to page coordinates and
de-duplicated across seams. Running a whole page through in one shot would shrink
the dots and find almost nothing.

In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO

TEST_DIR = DRIVE_ROOT / "test_images"
TEST_DIR.mkdir(exist_ok=True)
CONF = 0.20

candidates = sorted(
    list(TEST_DIR.glob("*.jpg")) + list(TEST_DIR.glob("*.png")) + list(TEST_DIR.glob("*.jpeg"))
)
if not candidates:
    print("No test images yet. Upload one to:", TEST_DIR)
else:
    TEST_IMAGE = candidates[0]
    print("Infer:", TEST_IMAGE)

    model = YOLO(str(BEST_PT))
    bgr = cv2.imread(str(TEST_IMAGE))
    h, w = bgr.shape[:2]
    stride = max(TILE - OVERLAP, 1)

    dets = []
    for oy in _origins(h, TILE, stride):
        for ox in _origins(w, TILE, stride):
            crop = bgr[oy : oy + min(TILE, h), ox : ox + min(TILE, w)]
            r = model.predict(
                source=crop, conf=CONF, imgsz=TILE, max_det=1000, verbose=False
            )
            if not r or r[0].boxes is None or len(r[0].boxes) == 0:
                continue
            for box, cf in zip(r[0].boxes.xyxy.cpu().numpy(), r[0].boxes.conf.cpu().numpy()):
                x0, y0, x1, y1 = box
                dets.append(((x0 + ox, y0 + oy, x1 + ox, y1 + oy), float(cf)))

    # suppress duplicates from overlapping seams
    kept = []
    if dets:
        dets.sort(key=lambda d: -d[1])
        centers = np.array([[(b[0] + b[2]) / 2, (b[1] + b[3]) / 2] for b, _ in dets])
        taken = np.zeros(len(dets), dtype=bool)
        for i in range(len(dets)):
            if taken[i]:
                continue
            kept.append(dets[i])
            d = np.linalg.norm(centers - centers[i], axis=1)
            taken |= d < 5.0
            taken[i] = True

    print(f"Detections on full page: {len(kept)}")

    vis = bgr.copy()
    for (x0, y0, x1, y1), cf in kept:
        cv2.rectangle(vis, (int(x0), int(y0)), (int(x1), int(y1)), (0, 220, 80), 1)

    out_dir = DRIVE_ROOT / "predict"
    out_dir.mkdir(exist_ok=True)
    out_path = out_dir / f"{TEST_IMAGE.stem}_dots.png"
    cv2.imwrite(str(out_path), vis)
    print("Overlay saved ->", out_path)

## 7) After training — copy weights to your PC

From Drive download:

```
MyDrive/BrailleLens_YOLO26/runs/detect/braille_dot_yolo26_tiled/weights/best.pt
```

Save locally as:

```
BrailleLens/yolo_dot_detect/runs/detect/braille_dot_yolo26_tiled/weights/best.pt
```

Then on your PC (`--tile 640` matches how the model was trained):

```bash
py -3.11 -m pip install -U ultralytics
py -3.11 -m yolo_dot_detect.infer --image test-img.jpeg --tile 640 --cluster
```

### If Colab disconnects mid-train
1. New runtime → GPU → remount Drive  
2. Re-run cells 1, 2, 3, 3b (tiles live on local disk, so they are rebuilt)  
3. Re-run **Train** — it resumes from `last.pt` on Drive